# Intro to SQLAlchemy-Core (Advanced)
This file shows the real power of SQLAlchemy. Object Oriented Programming!

In [ ]:
from os import getenv
from dotenv import load_dotenv
from sqlalchemy import ForeignKey, String, Row, Engine, create_engine, select, text
from sqlalchemy.orm import Session, DeclarativeBase, Mapped, mapped_column

In [ ]:
SQL_DIALECT = "postgresql"
DIALECT_LIBRARY = "psycopg2"

# TODO: Create a .env that contains your psql username and password
load_dotenv()
USER = getenv("USER")
PASSWORD = getenv("PASSWORD")

# TODO: Create a new database using psql named "test_sqlalchemy"
DATABASE = "test_sqlalchemy"

In [ ]:
engine = create_engine(f"{SQL_DIALECT}+{DIALECT_LIBRARY}://{USER}:{PASSWORD}@localhost/{DATABASE}")

In [ ]:
def reset_database() -> None:
    """Runs the schema.sql file to reset the database"""

    with open("schema.sql", "r", encoding="UTF-8") as schema:
        stmt = schema.read()

    with engine.connect() as conn:
        conn.execute(text(stmt))
        conn.commit()

reset_database()

As you can see, set up is very much the same as before, but we've imported a few more things that we will make big use of. We've imported the DeclarativeBase class, which we will inherit in our own base class. This is used to define classes that map to our tables, each row becomes an object!

In [ ]:
# We need to use inheritance to define a Base class for our tables,
# for now it doesn't need to do anything
class Base(DeclarativeBase):
    pass

# Now we define each table as a class, inheriting from the Base class we defined.


class Lecturer(Base):
    # Define the table
    __tablename__ = "lecturer"

    # Define the columns
    lecturer_id: Mapped[int] = mapped_column(primary_key=True)
    lecturer_name: Mapped[str] = mapped_column(String(30))


class Course(Base):
    __tablename__ = "course"

    course_id: Mapped[int] = mapped_column(primary_key=True)
    course_name: Mapped[str] = mapped_column(String(30))


class Student(Base):
    __tablename__ = "student"

    student_id: Mapped[int] = mapped_column(primary_key=True)
    student_name: Mapped[str] = mapped_column(String(30))


class StudentCourse(Base):
    __tablename__ = "student_course"

    student_course_id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("course.course_id"))
    student_id: Mapped[int] = mapped_column(ForeignKey("student.student_id"))

With our classes defined, we can start making functions that utilize them to access our database!

In [ ]:
def insert_lecturer(name: str):
    """Easy insert with no SQL code necessary"""
    with Session(engine) as session:
        lecturer = Lecturer(lecturer_name=name)
        session.add(lecturer)
        session.commit()


def insert_lecturers(names: list[str]):
    """Just pass a list of names and this will insert a new lecturer for each one!"""
    with Session(engine) as session:
        lecturers = [Lecturer(lecturer_name=name)
                     for name in names]
        session.add_all(lecturers)
        session.commit()


def get_by_key(table: type, key: int):
    with Session(engine) as session:
        row = session.get(table, key)
    return row


def select_lecturer_by_name(name: str):
    with Session(engine) as session:
        # Build an SQL statement out of python code
        stmt = select(Lecturer).where(Lecturer.lecturer_name == "SigmaBot")

        # We can use .scalar to run our statement
        by_name = session.scalar(stmt)

    return by_name

In [ ]:
insert_lecturer("SigmaBot")
#get_by_key(Lecturer, 1).lec
print(select_lecturer_by_name("SigmaBot").lecturer_name)

I won't be able to cover all methods, but it's very easy to query our database without writing any SQL!

In [ ]:
insert_lecturer("Siggy")
insert_lecturer("Ciggy??")

with Session(engine) as session:
    rows = session.query(Lecturer)

for row in rows:
    print(row.lecturer_name)

In [ ]:
with Session(engine) as session:
    count = session.query(Lecturer).count()

print(count)

We can use this to construct queries!

In [ ]:
with Session(engine) as session:
    query = session.query(Lecturer).filter(Lecturer.lecturer_name=="SigmaBot")

print(query)